In [1]:
"""
This notebook creates the dimension tables of our data model. This tables are not expected to change, so we only create them once using this notebook
and in theory it should never be executed again unless the ISTAC changes something like a code or the name of some airport
"""

'\nThis notebook creates the dimension tables of our data model. This tables are not expected to change, so we only create them once using this notebook\nand in theory it should never be executed again unless the ISTAC changes something like a code or the name of some airport\n'

In [2]:
import pandas as pd
import utils as u

# Airport

In [3]:
url_passengers = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000001/~latest.csv?lang=en"
url_gm = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000002/~latest.csv"
url_op = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000003/~latest.csv"

In [4]:
data_t = pd.read_csv(u.get_data_from_API_call(url_passengers))

In [5]:
df = data_t.copy(deep=True)

In [6]:
df.drop(columns=df.columns[df.columns.str.endswith('#es')], inplace=True)

In [7]:
df['AEROPUERTO_ESCALA_CODE']

0               CV
1               CV
2               CV
3               CV
4               CV
            ...   
3101035    GB_EGPF
3101036    GB_EGPF
3101037    GB_EGPF
3101038    GB_EGPF
3101039    GB_EGPF
Name: AEROPUERTO_ESCALA_CODE, Length: 3101040, dtype: object

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3101040 entries, 0 to 3101039
Data columns (total 16 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   SERVICIO_AEREO#en                object 
 1   SERVICIO_AEREO_CODE              object 
 2   MEDIDAS#en                       object 
 3   MEDIDAS_CODE                     object 
 4   MOVIMIENTO_AERONAVE#en           object 
 5   MOVIMIENTO_AERONAVE_CODE         object 
 6   TIME_PERIOD#en                   object 
 7   TIME_PERIOD_CODE                 object 
 8   AEROPUERTO_BASE#en               object 
 9   AEROPUERTO_BASE_CODE             object 
 10  AEROPUERTO_ESCALA#en             object 
 11  AEROPUERTO_ESCALA_CODE           object 
 12  OBS_VALUE                        float64
 13  ESTADO_OBSERVACION#en            object 
 14  ESTADO_OBSERVACION_CODE          object 
 15  CONFIDENCIALIDAD_OBSERVACION#en  float64
dtypes: float64(2), object(14)
memory usage: 378.5+ MB


Steps to getting the airports right:
1) Delete countries, autonomous communities and "Rest of"/"Remain of"
2) Join with airports from ourairports using word similitude or something like that

[Understanding _CODE from airports](https://www3.gobiernodecanarias.org/aplicaciones/appsistac/activos-semanticos/codelists/codelists/ISTAC/CL_AEROPUERTOS/01.004/detail)

I think I can do all I did (selecting only airports) with the API, with the "granularity" option, tomorrow ill check

Hello, it is tomorrow, it can not be done, I tried

In [9]:
# Extract countries
countries = df.loc[df['AEROPUERTO_ESCALA_CODE'].str.match(r'^[A-Z]{2}$', na=False), ['AEROPUERTO_ESCALA#en', 'AEROPUERTO_ESCALA_CODE']]
countries.drop_duplicates(inplace=True)
countries.rename({'AEROPUERTO_ESCALA#en': 'CountryName', 'AEROPUERTO_ESCALA_CODE': 'iso_country'}, inplace=True, axis=1)

In [10]:
countries

,CountryName,iso_country
0,Cabo Verde,CV
3504,Iceland,IS
9636,France,FR
18688,Spain,ES
20148,Ireland,IE
20440,Mauritania,MR
21608,Finland,FI
22192,Hungary,HU
26572,Poland,PL
28032,Estonia,EE


In [11]:
# Rest of/Remain of AEROPUERTO_ESCALA#en have "_O" at the end of their code
df_f = df.loc[~df['AEROPUERTO_ESCALA_CODE'].str.endswith('_O')]

# Delete entries with the whole country
df_f = df_f.loc[~df_f['AEROPUERTO_ESCALA_CODE'].str.match(r'^[A-Z]{2}$', na=False)]

# Delete autonomous communities (Their code is like ES[0-9][0-9]) 
df_f = df_f.loc[~df_f['AEROPUERTO_ESCALA_CODE'].str.match(r'^ES[0-9]{2}$', na=False)]

# Delete sum of entire island
df_f = df_f.loc[~df_f['AEROPUERTO_ESCALA_CODE'].str.match(r'^ES70[0-9]$', na=False)]

# Delete sum of all autonomous communities and sum of all islands
df_f = df_f.loc[~((df_f['AEROPUERTO_ESCALA_CODE'] == 'ES_XES70') | (df_f['AEROPUERTO_ESCALA_CODE'] == 'ES70') | (df_f['AEROPUERTO_ESCALA_CODE'] == 'FOREIGN'))]

In [12]:
df_f.loc[~df_f['AEROPUERTO_ESCALA#en'].str.endswith('Airport', na=False), 'AEROPUERTO_ESCALA#en'].unique()

array(['Trondheim Airport Vèrnes', 'Bergen Airport Flesland',
       'Zaragoza Air Base', 'Sandefjord Airport, Torp',
       'Harstad/Narvik Airport, Evenes', 'Stavanger Airport Sola',
       'Amsterdam Airport Schiphol', 'Václav Havel Airport Prague',
       'Moss Airport, Rygge'], dtype=object)

In [13]:
istac_airports = df_f[['AEROPUERTO_ESCALA#en', 'AEROPUERTO_ESCALA_CODE']].copy(deep=True)

In [14]:
istac_airports.drop_duplicates(inplace=True)

In [15]:
istac_airports

,AEROPUERTO_ESCALA#en,AEROPUERTO_ESCALA_CODE
292,Nouadhibou International Airport,MR_GQPP
584,Brussels Airport,BE_EBBR
876,Durham Tees Valley Airport,GB_EGNV
1168,Dublin Airport,IE_EIDW
1460,Växjö Kronoberg Airport,SE_ESMX
...,...,...
84680,Eelde Airport,NL_EHGG
84972,Ålesund Airport,NO_ENAL
85264,Tromsô Airport,NO_ENTC
85556,Nantes Atlantique Airport,FR_LFRS


In [16]:
istac_airports['ident'] = istac_airports['AEROPUERTO_ESCALA_CODE'].str[3:]

In [18]:
airport_csv = pd.read_csv('airports.csv')

In [19]:
airport_csv = airport_csv.merge(countries, on='iso_country')

In [20]:
join = istac_airports.merge(airport_csv, on='ident', suffixes=("l", "r"))

In [21]:
join

,AEROPUERTO_ESCALA#en,AEROPUERTO_ESCALA_CODE,ident,id,type,name,latitude_deg,longitude_deg,elevation_ft,continent,...,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords,CountryName
0,Nouadhibou International Airport,MR_GQPP,GQPP,3138,large_airport,Nouadhibou International Airport,20.932404,-17.030199,24.0,AF,...,Nouadhibou,yes,GQPP,NDB,GQPP,NaN,NaN,https://en.wikipedia.org/wiki/Nouadhibou_Inter...,NaN,Mauritania
1,Brussels Airport,BE_EBBR,EBBR,2155,large_airport,Brussels Airport,50.901402,4.484440,175.0,EU,...,Zaventem,yes,EBBR,BRU,EBBR,NaN,http://www.brusselsairport.be/en/,https://en.wikipedia.org/wiki/Brussels_Airport,"Brussels National, Zaventem, EBMB",Belgium
2,Durham Tees Valley Airport,GB_EGNV,EGNV,2449,medium_airport,Teesside International Airport,54.509201,-1.429410,120.0,EU,...,"Darlington, Durham",yes,EGNV,MME,EGNV,NaN,https://www.teessideinternational.com/,https://en.wikipedia.org/wiki/Teesside_Interna...,"Durham Tees Valley Airport, RAF Middleton St G...",United Kingdom of Great Britain and Northern I...
3,Dublin Airport,IE_EIDW,EIDW,2533,large_airport,Dublin Airport,53.428713,-6.262121,242.0,EU,...,Dublin,yes,EIDW,DUB,EIDW,NaN,http://www.dublinairport.com/,https://en.wikipedia.org/wiki/Dublin_Airport,Aerfort Bhaile Átha Cliath,Ireland
4,Växjö Kronoberg Airport,SE_ESMX,ESMX,2672,medium_airport,Växjö Kronoberg Airport,56.929100,14.728000,610.0,EU,...,Växjö,yes,ESMX,VXO,ESMX,NaN,NaN,https://en.wikipedia.org/wiki/V%C3%A4xj%C3%B6_...,NaN,Sweden
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,Eelde Airport,NL_EHGG,EHGG,2519,large_airport,Groningen Airport Eelde,53.119107,6.577652,17.0,EU,...,Groningen,yes,EHGG,GRQ,EHGG,NaN,NaN,https://en.wikipedia.org/wiki/Groningen_Airpor...,NaN,Netherlands
191,Ålesund Airport,NO_ENAL,ENAL,2564,large_airport,Ålesund Airport,62.560443,6.110845,69.0,EU,...,Ålesund,yes,ENAL,AES,ENAL,NaN,https://www.avinor.no/airport/alesund/,https://en.wikipedia.org/wiki/%C3%85lesund_Air...,"Vigra, Ålesund Vigra Airport",Norway
192,Tromsô Airport,NO_ENTC,ENTC,2599,large_airport,Tromsø Airport,69.683296,18.918900,31.0,EU,...,Tromsø,yes,ENTC,TOS,ENTC,NaN,http://www.avinor.no/en/airport/tromso,https://en.wikipedia.org/wiki/Troms%C3%B8_Airport,"Langnes, Tromsøya, Tromso",Norway
193,Nantes Atlantique Airport,FR_LFRS,LFRS,4221,medium_airport,Nantes Atlantique Airport,47.153198,-1.610730,90.0,EU,...,Nantes,yes,LFRS,NTE,LFRS,NaN,https://www.nantes.aeroport.fr/en,https://en.wikipedia.org/wiki/Nantes_Atlantiqu...,NaN,France


Only two missing airports, I can input them by hand

In [22]:
print(f"Mising airports: {len(istac_airports['AEROPUERTO_ESCALA#en'].unique()) - len(join['AEROPUERTO_ESCALA#en'].unique())}")
missing_airports = set(istac_airports['AEROPUERTO_ESCALA#en'].unique()) - set(join['AEROPUERTO_ESCALA#en'].unique())
print(f"Missing airports: {missing_airports}")

Mising airports: 4
Missing airports: {'Dakhla Airport', 'Berlin-Tegel Airport', 'Robin Hood Doncaster Sheffield Airport', 'Hassan I Airport'}


In [23]:
result_df = join[['AEROPUERTO_ESCALA#en', 'AEROPUERTO_ESCALA_CODE', 'latitude_deg', 'longitude_deg','iso_country', 'CountryName']].copy(deep=True)
result_df.rename({'iso_country': 'CountryCode', 'AEROPUERTO_ESCALA#en': 'AirportName', 
                  'latitude_deg': 'Latitude', 'longitude_deg': 'Longitude', 
                  'country': 'CountryName', 'AEROPUERTO_ESCALA_CODE': 'AirportCode'}, axis=1, inplace=True)

In [24]:
result_df['AirportId'] = result_df.index

In [25]:
result_df.to_csv('../../data/Airport.csv', index=False)
result_df

,AirportName,AirportCode,Latitude,Longitude,CountryCode,CountryName,AirportId
0,Nouadhibou International Airport,MR_GQPP,20.932404,-17.030199,MR,Mauritania,0
1,Brussels Airport,BE_EBBR,50.901402,4.484440,BE,Belgium,1
2,Durham Tees Valley Airport,GB_EGNV,54.509201,-1.429410,GB,United Kingdom of Great Britain and Northern I...,2
3,Dublin Airport,IE_EIDW,53.428713,-6.262121,IE,Ireland,3
4,Växjö Kronoberg Airport,SE_ESMX,56.929100,14.728000,SE,Sweden,4
...,...,...,...,...,...,...,...
190,Eelde Airport,NL_EHGG,53.119107,6.577652,NL,Netherlands,190
191,Ålesund Airport,NO_ENAL,62.560443,6.110845,NO,Norway,191
192,Tromsô Airport,NO_ENTC,69.683296,18.918900,NO,Norway,192
193,Nantes Atlantique Airport,FR_LFRS,47.153198,-1.610730,FR,France,193


# Territory, AircraftMovement, AirService

In [26]:
url = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000013/~latest.csv?lang=en"

In [27]:
data = pd.read_csv(u.get_data_from_API_call(url))

In [28]:
df = data.copy(deep=True)

In [29]:
df_terr = pd.concat([df[['TERRITORIO_CODE', 'TERRITORIO#en']], df[['AEROPUERTO_ESCALA_CODE', 'AEROPUERTO_ESCALA#en']].rename({'AEROPUERTO_ESCALA#en': 'TERRITORIO#en', 'AEROPUERTO_ESCALA_CODE': 'TERRITORIO_CODE'}, axis=1)]).drop_duplicates()

df_terr.reset_index(inplace=True, drop=True)

# Remove "Total", "Foreign and Spain (Canary Islands excluded)", "Spain"
df_terr = df_terr.loc[~df_terr['TERRITORIO_CODE'].isin(["_T_XES70", "ES", "_T"])]

df_terr['TerritoryId'] = df_terr.index

df_terr.rename({'TERRITORIO_CODE': 'TerritoryCode', 'TERRITORIO#en': 'TerritoryName'}, inplace=True, axis=1)

df_terr = df_terr[['TerritoryId', 'TerritoryCode', 'TerritoryName']]

df_terr.to_csv('../../data/Final_Territory.csv', index=False)

In [30]:
df_am = df[['MOVIMIENTO_AERONAVE_CODE', 'MOVIMIENTO_AERONAVE#en']].drop_duplicates().reset_index(drop=True)

df_am['AircraftMovementId'] = df_am.index

df_am.rename({'MOVIMIENTO_AERONAVE_CODE': 'AircraftMovementCode', 'MOVIMIENTO_AERONAVE#en': 'AircraftMovement'}, axis=1, inplace=True)

df_am = df_am[['AircraftMovementId', 'AircraftMovementCode', 'AircraftMovement']]

df_am.to_csv('../../data/Final_AircraftMovement.csv', index=False)

In [31]:
df_as = df[['SERVICIO_AEREO_CODE', 'SERVICIO_AEREO#en']].drop_duplicates().reset_index(drop=True)

df_as['AirServiceId'] = df_as.index

df_as.rename({'SERVICIO_AEREO_CODE': 'AirServiceCode', 'SERVICIO_AEREO#en': 'AirService'}, axis=1, inplace=True)

df_as = df_as[['AirServiceId', 'AirServiceCode','AirService', ]]

df_as.to_csv('../../data/AirService.csv', index=False)